# Introduction

Generation of the artificial mass spectra is demostrated.
Generated spectra are required to validate peak fitting performance.
The advantage of using simulated spectra for validations is known true values. 

# Requirements

We will consider two scenarios.

**Scenario 1**
Ideal peak shape and resolution function.

Vary:
* number of peaks
* position of peaks
* intensity of peaks
* (peak shape)
* (resolution)

**Scenario 2**
Distorted signal (random noise, skewed peak shape). Peak shape is a probability distribution, tests using different numbers of samples.

Additionally vary:
* noise level
* skewness of peaks (difference of synthesized peak compared to assumed peak shape)

Scenario 1 is what we observe in Orbitrap spectra, Scenario 2 is closer to TOF. 

# Modules

In [ ]:
import numpy as np
from numba import jit
from scipy.stats import norm, skewnorm, poisson
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from mascope_file.io import load_coord
from mascope_signal.compute import get_sum_signal
from mascope_signal.peak import segment_spec, fit_n_peaks
from functools import partial
import warnings

plotly_config = {
  'toImageButtonOptions': {
    'format': 'png',
    'scale':2
  }
}

In [ ]:
def find_closest_indices(detected, ground_truth):
    """
    Finds the closest indices of values from 'detected' in 'ground_truth'.

    Parameters:
    detected (np.array): Array of detected values.
    ground_truth (np.array): Array of ground truth values.

    Returns:
    np.array: Array of indices in 'ground_truth' that are closest to each value in 'detected'.
    """
    # Create an array to hold the indices of the closest values
    detected = np.asarray(detected)
    closest_indices = np.zeros_like(detected, dtype=int)
    
    for i, value in enumerate(detected):
        # Find the index of the closest value in ground_truth
        closest_index = np.abs(ground_truth - value).argmin()
        closest_indices[i] = closest_index
    
    return closest_indices

# Peak shape

The module `scipy.stats` will be used for generation of peak shapes.

Orbitrap peaks are generally symmetrical and have a consistent shape across the mass range.
The Gaussian distribution `scipy.stats.norm` accurately models this behavior.

Time-of-Flight (TOF) mass spectrometers often produce slightly asymmetrical peaks with a tail on the higher mass side.
For this type of spectrum we will consider a skew-normal distribution `scipy.stats.skewnorm`.
An exponentially modified Gaussian distribution `scipy.stats.exponnorm` can also be suitable for modeling TOF peaks with a slight tail.

In [ ]:
def gen_peak_shape(x, ms, scale=1, a=1):
    """Generate peak shape as dict"""
    match ms:
        case "orbi":
            y = norm.pdf(x, scale=scale)
        case "tof":
            y = skewnorm.pdf(x, scale=scale, a=a)
        case _:
            raise ValueError("Unknown mass spectrometer. Choose orbi or tof.")
    peak_shape = {"x": x, "y": y}
    return peak_shape

In [ ]:
x = np.linspace(-5, 5, 100)

fig = plt.figure(figsize=(7, 3))
plt.plot(x, skewnorm.pdf(x, a=2), label="TOF peak shape example")
plt.plot(x, norm.pdf(x), label="Orbi peak shape example")
plt.legend();

In [ ]:
@jit
def generate_gaussian_peaks(x, means, sigmas, heights):
    """Add Gaussian peaks with specified parameters to spectrum"""
    num_peaks = len(means)
    spec = np.zeros_like(x)
    for i in range(num_peaks):
        spec += heights[i] * np.exp(-0.5 * ((x - means[i]) / sigmas[i]) ** 2)
    return spec

In [ ]:
def generate_skewed_gaussian_peaks(x, means, sigmas, heights, a=2):
    """Add skewed Gaussian peaks with specified parameters to spectrum"""
    num_peaks = len(means)
    spec = np.zeros_like(x)
    for i in range(num_peaks):
        # Generate skewed Gaussian distribution
        skewnorm_pdf = skewnorm.pdf(x, a, loc=means[i], scale=sigmas[i])
        spec += heights[i] * skewnorm_pdf / skewnorm_pdf.max()
    return spec

# Resolution function

TOF resolution function ([Junninen, 2013](https://helda.helsinki.fi/server/api/core/bitstreams/6b7681ca-5529-4c82-bc86-de670fbaf77f/content)):

$$R(mz) = \frac{mz}{a\cdot mz + b}$$

Orbitrap resolution function ([Perry et al, 2008](https://doi.org/10.1002/mas.20186)):

$$R(mz) = \frac{a}{\sqrt{mz}}$$

where $a$ and $b$ are fitting coefficients.

In [ ]:
def res_fun_orbi(mz, a=1.725e6):
    return a / np.sqrt(mz)

def res_fun_tof(mz, a, b):
    return mz / (a * mz + b)

# M/Z grid

Knowing the resolution, we can precompute m/z grid.



In [ ]:
@jit(nopython=True)
def precompute_grid(mz_min, mz_max, ms, points_per_fwhm=4, res_coef=None):
    """Precompute mz grid based on the resolution function: 
        Orbi: a / sqrt(mz)
        TOF: mz / (a*mz + b)

    :param points_per_fwhm: number of data points per FWHM of the peak
    :type points_per_fwhm: float, optional
    :param resolution_coeff: Resolution function coefficient
    :type resolution_coeff: float, optional, defaults to None
    :return: computed mz grid
    :rtype: numpy.ndarray
    """
    if res_coef == None:
        if ms == "orbi":
            res_coef = (1.715e6, 0.0)
        if ms == "tof":
            res_coef = (1e-4, 1e-3)
    # Set starting mz value
    mz = mz_min
    # Initialize list with mz grid
    mz_grid = [mz_min]
    while mz < mz_max:
        if ms == "orbi":
            resolution = res_coef[0] / np.sqrt(mz)
        if ms == "tof":
            resolution = mz / (res_coef[0] * mz + res_coef[1])
        fwhm = mz / resolution
        # Step to the next point of the grid
        step = fwhm / points_per_fwhm
        # Add a new point to the mz grid
        mz += step
        mz_grid.append(mz)

    return np.array(mz_grid, dtype=np.float32)

# Noise

Orbitrap spectra do not contain noise due to build-in preprocessing and thresholding.

The noise in TOF spectra will be modeled with Poisson distribution ([Cubison et al, 2014](https://amt.copernicus.org/preprints/7/12617/2014/amtd-7-12617-2014.pdf)) using `scipy.stats.poisson`.

In [ ]:
def add_poisson_noise(spec):
    """Adds the poisson noise to the spectrum, return spectrum with a noise"""
    noise = poisson.rvs(mu=np.sqrt(spec).mean(), size=spec.shape)
    return spec + noise

# Metrics

To measure the "goodness" of the fit the following metrics can be used:
1. **Coefficient of Determination ($R^2$)**. This metric measures the proportion of variation in the data that is explained by the model. An $R^2$ value close to 1 indicates a good fit.
2. **Residual Sum of Squares (RSS)**. This is a common metric used to evaluate the quality of a fit. RSS is calculated as the sum of the squared differences between the observed data points and the fitted curve. A lower RSS value indicates a better fit.
3. **Chi-squared χ²**. It is a measure used in statistics to assess the goodness of fit between an observed data set and a theoretical distribution. It is particularly useful in hypothesis testing and in evaluating how well a model fits the data. χ² is among fit statistics values we get in lmfit.

To estimate overall m/z error we will use the root mean squared error (RMSE).
It measures the average of the differences between observed and predicted values.
We will also get an array of differences in ppm.

The intensity of the peaks may vary drammatically, so as the difference between predicted and observed values. The errors in absolute values may not be representative in this case. We will use (mean) absolute **percentage** error (MAPE/APE) measuring the average of the absolute percentage differences between observed and predicted intensities.

We will also calculate the difference in number between predicted and observed number of peaks.

In [ ]:
def calculate_r_squared(observed, predicted):
    """
    Calculate the R-squared value for observed and predicted values.

    Parameters:
    observed (np.array): Array of observed (actual) values.
    predicted (np.array): Array of predicted values.

    Returns:
    float: metric value.
    """
    # Calculate the mean of the observed values
    mean_observed = np.mean(observed)
    # Calculate the total sum of squares (TSS)
    tss = np.sum((observed - mean_observed) ** 2)
    # Calculate the residual sum of squares (RSS)
    rss = np.sum((observed - predicted) ** 2)
    # Calculate R-squared
    r_squared = 1 - (rss / tss)
    return r_squared

In [ ]:
def calculate_rss(observed, predicted):
    """
    Calculate the Residual Sum of Squares (RSS) between observed and predicted values.

    Parameters:
    observed (numpy.ndarray): The observed values.
    predicted (numpy.ndarray): The predicted values.

    Returns:
    float: The Residual Sum of Squares.
    """
    residuals = observed - predicted
    rss = np.sum(residuals ** 2)
    return rss

In [ ]:
def calculate_rmse(observed, predicted):
    return np.sqrt(np.mean((observed - predicted) ** 2))

In [ ]:
def calculate_ppm_error(observed, predicted):
    """Calculate the difference between observed and predicted m/z values in ppm."""
    return abs(observed - predicted) / predicted * 10**6

In [ ]:
def calculate_ape(observed, predicted, mean=True):
    """Calculate absolute percentage error (mean if needed)"""
    if mean:
        return np.mean(np.abs((observed - predicted) / observed)) * 100
    else:
        return np.abs((observed - predicted) / observed) * 100

In [ ]:
def create_error_plot(fitted_pos, fitted_hei, positions, heights):
    # Indices of GT values closest to those in fitted_pos
    close_pos_inds = find_closest_indices(fitted_pos, positions)

    # Get GT values corresponding to fitted values
    observed_pos = positions[close_pos_inds]
    observed_hei = heights[close_pos_inds]

    # ppm difference
    ppm_err = calculate_ppm_error(observed_pos, fitted_pos)

    # Overall m/z error
    mz_err = calculate_rmse(observed_pos, fitted_pos)

    # height error
    hei_err = calculate_ape(observed_hei, fitted_hei, mean=False)
    # mean height error
    mean_hei_err = calculate_ape(observed_hei, fitted_hei)

    fig = make_subplots(rows=1, cols=2, subplot_titles=[f"RMSE={mz_err:.2e} ppm", f"MAPE={mean_hei_err:.2f}%"])

    fig.add_trace(go.Scatter(x=observed_pos, y=ppm_err, mode="markers", name="ppm error"), row=1, col=1)
    fig.add_trace(go.Scatter(x=observed_pos, y=hei_err, mode="markers", name="Height error"), row=1, col=2)

    fig.update_layout(
        yaxis_title="Error, ppm",
        xaxis_title="M/z of the detected peak, Th",
        yaxis2_title="Error, %",
        xaxis2_title="M/z of the detected peak, Th",
        margin=dict(l=10, r=10, t=40, b=10),
        width=900, 
        height=300
    )

    return fig

# Data variation

In [ ]:
def vary_var(rng, var, val=5):
    """Return random value in range var+-val using randomiser rng. val in %"""
    var *= (1 + rng.uniform(-val/100, val/100))
    return var

# Investigate Orbitrap spectra

The spectrum will be generated as segments, each is a pair of counts and m/z values.
First it would be helpful to investigate the real spectra.

In [ ]:
real_filename = "KORBI2_20240626_NG_165401_-"

# Get sum signal and mz coordinate
real_sum_sig = get_sum_signal(real_filename).values
real_mz = load_coord(real_filename, "sum_signal", "mz")

# Segment sum signal and get segments inds
real_seg_indices = segment_spec(real_sum_sig)

# Get final segments
real_seg_signal = [real_sum_sig[chunk] for chunk in real_seg_indices]
real_seg_mz = [real_mz[chunk] for chunk in real_seg_indices]

## Segment length statistics

In [ ]:
def inverse_x(x, a, b):
    """Custom function for 1/x"""
    return a / (x + b)

In [ ]:
chunk_lengths = [len(i) for i in real_seg_mz]

# Get distribution of lengths
bins, edges = np.histogram(chunk_lengths, bins=10)

# Fit inverse x function to the distribution of segment lengths
popt, pcov = curve_fit(inverse_x, edges[1:], bins / max(bins))

# Sample segment lengths
x_range = np.linspace(min(edges), max(edges), 1)
uniform_samples = np.random.uniform(
    low=min(edges), high=max(edges), size=len(real_sum_sig)
)
uniform_samples = uniform_samples.astype("int")

# Get a histogram of uniformely distributed lengths
gen_bins, gen_edges = np.histogram(uniform_samples, bins=10)

# Apply inverse distribution
samples = gen_bins * inverse_x(gen_edges[1:], popt[0], popt[1])

fig = plt.figure(figsize=(10, 3))
plt.scatter(x=edges[1:], y=bins, label="Real sample lengths")
plt.scatter(x=gen_edges[1:], y=samples, label="Generated sample lengths")
plt.legend()

It would also be interesting to look at the mz dependence of segment length. 

In [ ]:
segment_mean_mzs = [np.mean(i) for i in real_seg_mz]

fig = plt.figure(figsize=(10, 3))
plt.scatter(x=segment_mean_mzs, y=chunk_lengths)

From the figure we confirm that the most frequent segment length is seven, and segment lengths are more or less evenly distributed (taking into account, that we don't have peaks everywhere).

## Number of peaks per segment

To quickly estimate the max number of peaks per sergment, we will use `find_peaks` function from `scipy.signal`. 

In [ ]:
from scipy.signal import find_peaks

detected_peaks_per_segment = [len(find_peaks(i)[0]) for i in real_seg_signal]

fig = plt.figure(figsize=(10, 3))
plt.hist(detected_peaks_per_segment, bins=3)
plt.yscale("log")

In [ ]:
np.where(np.array(detected_peaks_per_segment) == 1)[0]

fig = plt.figure(figsize=(10, 3))
plt.plot(real_seg_mz[1170], real_seg_signal[1170], marker="o")
plt.grid()

As expected, major segments contain just one peak since Orbitrap spectra possess great resolution.
All the segments with three peaks look like a noise.
Among segments containing two peaks only in one the number of peaks was actually two.

## Peak heights

Peak intensity varies from 1e1 to 1e6 counts.
The intensity should not affect the fitting in case of orbitrap that much since we usually have just one peak per spectrum and this peak is normalized during the fitting.
However, we still stick to this range while segments.

The range of heights will be modelled using the power distribution.

# Orbitrap pipeline

The initial task was to generate spectrum segments.
Apparently, this is way less efficient compared to the generation of the whole spectrum for several reasons:
1. Precomputing the whole range of m/z values in one run is faster than computing it for each sector
2. We don't have to figure out the sector length, it comes naturally after segmentation of the whole spectrum.
3. We don't have to put overcomplicated limitations on a peak position within a sector so that the whole peak would fit in the sector (as we see it in real orbitrap spectra).
4. We can take advantage of vectorized operations while generating peak parameters. It makes calculations way faster.

In [ ]:
gen_params = {
    "ms": "orbi",
    "mzs": (100, 1000),
    "heis": (1e1, 1e6),  # Peak height range
    "lens": (6, 16),  # Sector length range
    "popt": (3.088, 1.844),  # Sector length distribution params
    "points_per_fwhm": 4,  # Points per FWHM (in mz scale computation)
    "num_of_peaks": 100, # NUmber of peaks to generate
    "res_fun_coef": 1.715e6 # Resolution function coefficient
}

# Precompute sigma multiplier for peak generation
SIGMA_MULTIPLIER = 2 * np.sqrt(2 * np.log(2))

In [ ]:
## Generate peak positions and heights

# Set randomizer
rng = np.random.default_rng(33)

# Slightly change resolution coefficient
res_coef = vary_var(rng, gen_params["res_fun_coef"])
res_coef = (res_coef, 0.0) # for numba to work properly

# Precompute mz range
mz_grid = precompute_grid(
    min(gen_params["mzs"]),
    max(gen_params["mzs"]),
    ms=gen_params["ms"],
    points_per_fwhm=gen_params["points_per_fwhm"],
    res_coef=res_coef
)

# Generate peak positions
positions = rng.uniform(min(mz_grid), max(mz_grid), gen_params["num_of_peaks"])

# Generate peak heights, power  distribution
heights = rng.power(0.01, positions.size)
# Scale heights to proper range
heights = np.interp(
    heights,
    (heights.min(), heights.max()),
    (gen_params["heis"][0], gen_params["heis"][1]),
)

# Calculate resolution for each datapoint
resolutions = res_fun_orbi(positions, res_coef[0])

# Standard deviation for Gaussian distribution
sigmas = positions / resolutions / SIGMA_MULTIPLIER

# plt.vlines(positions, ymin=0, ymax=heights);

In [ ]:
# Get final artificial spectrum
art_spec = generate_gaussian_peaks(mz_grid, positions, sigmas, heights)

In [ ]:
## Fit Orbitrap spectrum

# Segment artificial spectrum
segmented_indices = segment_spec(art_spec)

# Initialise fitted peak list, ignore RuntimeWarnings from lmfit, fit peaks
fitted_peaks = []
residuals = []
chisqr = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for segment in segmented_indices:
        fit, fitted_chunk = fit_n_peaks(
            mz_grid[segment],
            art_spec[segment],
            {"x": x, "y": norm.pdf(x)},
            partial(res_fun_orbi, a=res_coef[0]),
            0.9,
        )
        residuals += fit.residual.tolist()
        chisqr.append(fit.chisqr)
        for i in fitted_chunk:
            fitted_peaks.append(i)

# Convert fitted peaks to numpy array
fitted_peaks = np.asarray(fitted_peaks)
# Get fitted peak positions and heights
fitted_pos = fitted_peaks[:,0]
fitted_hei = fitted_peaks[:,1]

# Get convert residuals to numpy array and get mz scale for them
residuals = np.asarray(residuals)
mz_residuals = np.concatenate([mz_grid[segment] for segment in segmented_indices])

print(f"""
Initially we had {len(positions)} peaks
Mascope common fitting routine found {len(fitted_peaks)} peaks 
""")

In [ ]:
# Create a subplot with 2 rows and 1 column
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[1, 4]
)

# Add traces to the main plot (row 2, col 1)
fig.add_trace(
    go.Scatter(x=mz_grid, y=art_spec, name="Generated spectrum"), row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=positions,
        y=heights,
        mode="markers",
        marker=dict(size=7, symbol="square"),
        name="Ground Truth",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=fitted_pos, y=fitted_hei, mode="markers", name="Mascope detections"),
    row=2,
    col=1,
)

# Add trace for residuals (row 1, col 1)
fig.add_trace(
    go.Scatter(x=mz_residuals, y=residuals, mode="lines", name="Residuals"),
    row=1,
    col=1,
)

# Update layout
fig.update_layout(
    width=600,
    height=300,
    margin=dict(l=10, r=10, t=40, b=10),
    title=f"Orbi artificial spectrum. χ²={np.mean(chisqr):.2e} (less is better)",
    xaxis2_title="m/z, Th",
    yaxis2_title="Counts, a.u."
)

# Show the figure
fig.show(config=plotly_config)

In [ ]:
fig = create_error_plot(fitted_pos, fitted_hei, positions, heights)
fig.show(config=plotly_config)

In the figure above we clearly observe seven outliers, the peaks that were not supposed to be found in our signal.

# Investigate TOF spectra

Same as with Orbitrap spectra, we need to look at the real data to figure out how to model it

In [ ]:
real_filename = "KLTOF1_2023.12.05_13h41m32s_NG_IN_ACN_SAMPLE_-"

# Get sum signal and mz coordinate
real_sum_sig = get_sum_signal(real_filename).values
real_mz = load_coord(real_filename, "sum_signal", "mz")

In [ ]:
peak, _ = find_peaks(real_sum_sig, height=np.percentile(real_sum_sig, 99.))

fig = go.Figure([
    go.Scatter(x=real_mz, y=real_sum_sig, name="Real spectrum"),
    go.Scatter(x=real_mz[peak], y=real_sum_sig[peak], name="Real spectrum", mode="markers")
    ])
fig.show()

In [ ]:
plt.figure(figsize=(10,3))
plt.hist(real_sum_sig[peak], bins=10)
plt.yscale("log")

# TOF pipeline

In this case we will use the same approach as in the case of Orbitrap with several exceptions:
1. Peak shape is skewed Gaussian instead of purely Gaussian.
2. The Poisson noise will be added to the final signal.
3. Spectrum will be segmented by integer m/z values +- dmz, where dmz=0.5.

In [ ]:
gen_params = {
    "ms": "tof",
    "mzs": (100, 200),
    "heis": (1e1, 1e6),  # Peak height range
    "points_per_fwhm": 10,  # Points per FWHM (in mz scale computation)
    "num_of_peaks": 1000, # Number of peaks to generate
    "alpha": 2, # Skewness of the Gaussian peak
    "res_fun_coefs": (1e-4, 1e-3) # Resolution function coefficients
}

# Precompute sigma multiplier for peak generation
SIGMA_MULTIPLIER = 2 * np.sqrt(2 * np.log(2))

In [ ]:
# Set randomizer
rng = np.random.default_rng(33)

# Slightly change resolution coefficient (+-5%)
a_coef = vary_var(rng, gen_params["res_fun_coefs"][0])
b_coef = vary_var(rng, gen_params["res_fun_coefs"][1])

# Precompute mz range
mz_grid = precompute_grid(
    min(gen_params["mzs"]),
    max(gen_params["mzs"]),
    ms=gen_params["ms"],
    points_per_fwhm=gen_params["points_per_fwhm"],
    res_coef=(a_coef, b_coef),
)

# Generate peak positions
positions = rng.uniform(min(mz_grid), max(mz_grid), gen_params["num_of_peaks"])

# Generate peak heights, power  distribution
heights = rng.power(0.01, positions.size)
# Scale heights to proper range
heights = np.interp(
    heights,
    (heights.min(), heights.max()),
    (gen_params["heis"][0], gen_params["heis"][1]),
)

# Calculate resolution for each datapoint
resolutions = res_fun_tof(positions, a=a_coef, b=b_coef)

# Standard deviation for Gaussian distribution
sigmas = positions / resolutions / SIGMA_MULTIPLIER

# plt.vlines(positions, ymin=0, ymax=heights)

In [ ]:
# Tweak skewness
alpha = vary_var(rng, gen_params["alpha"])
# Generate artificial spectrum
art_spec = generate_skewed_gaussian_peaks(mz_grid, positions, sigmas, heights, a=alpha)

In [ ]:
# Add noise to artificial spectrum
art_spec_with_noise = add_poisson_noise(art_spec)

In [ ]:
fig = go.Figure(
    [
        go.Scatter(x=mz_grid, y=art_spec, name="Artificial spectrum"),
        go.Scatter(
            x=mz_grid, y=art_spec_with_noise, name="Artificial spectrum with noise"
        ),
    ]
)
fig.update_layout(
    height=300,
    width=800,
    margin=go.layout.Margin(l=30, r=30, b=30, t=40, pad=4),
    title="TOF artificial spectra comparison: clean vs noisy",
    xaxis_title="m/z, Th",
    yaxis_title="Counts, a.u",
)
fig.show()

We need at least 1000 peaks per 100 Th to have peak intersections in the artificial spectrum, similar to observed in live spectra.

Lower signals (<1000 counts) still have a bit of a difference compared to the real peaks if we don't try to generate huge amount of peaks (>1e5 per 100 Th).
Higher signals are almost indistiguishible from the real ones.
The background also looks the same.

In [ ]:
## Fit TOF spectrum

# Segment artificial spectrum
dmz = 0.5
u_list = np.arange(gen_params["mzs"][0], gen_params["mzs"][1] + 1)
specs_to_fit = [
    (
        mz_grid[np.logical_and(mz_grid >= u - dmz, mz_grid <= u + dmz)],
        art_spec_with_noise[np.logical_and(mz_grid >= u - dmz, mz_grid <= u + dmz)],
    )
    for u in u_list
]

# Initialise fitted peak list, ignore RuntimeWarnings from lmfit, fit peaks
fitted_peaks = []
residuals = []
chisqr = []
mz_residuals = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for i, j in specs_to_fit:
        fit, fitted_chunk = fit_n_peaks(
            i,
            j,
            {"x": x, "y": skewnorm.pdf(x, a=2)},
            partial(
                res_fun_tof,
                a=gen_params["res_fun_coefs"][0],
                b=gen_params["res_fun_coefs"][1],
            ),
            0.9,
        )
        if fit == None:
            continue
        residuals += fit.residual.tolist()
        chisqr.append(fit.chisqr)
        residuals += fit.residual.tolist()
        mz_residuals.append(i)
        chisqr.append(fit.chisqr)
        for i in fitted_chunk:
            fitted_peaks.append(i)

# Convert fitted peaks to numpy array
fitted_peaks = np.asarray(fitted_peaks)
# Get fitted peak positions and heights
fitted_pos = fitted_peaks[:,0]
fitted_hei = fitted_peaks[:,1]

# Get convert residuals to numpy array and get mz scale for them
residuals = np.asarray(residuals)
mz_residuals = np.concatenate(mz_residuals)

print(
    f"""
Initially we had {len(positions)} peaks
Mascope common fitting routine found {len(fitted_peaks)} peaks 
"""
)

In [ ]:
# Create a subplot with 2 rows and 1 column
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[1, 4]
)

# Add traces to the main plot (row 2, col 1)
fig.add_trace(
    go.Scatter(x=mz_grid, y=art_spec, name="Generated spectrum"), row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=positions,
        y=heights,
        mode="markers",
        marker=dict(size=7, symbol="square"),
        name="Ground Truth",
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=fitted_pos, y=fitted_hei, mode="markers", name="Mascope detections"),
    row=2,
    col=1,
)

# Add trace for residuals (row 1, col 1)
fig.add_trace(
    go.Scatter(x=mz_residuals, y=residuals, mode="lines", name="Residuals"),
    row=1,
    col=1,
)

# Update layout
fig.update_layout(
    width=600,
    height=300,
    margin=dict(l=10, r=10, t=40, b=10),
    title=f"TOF artificial spectrum. χ²={np.mean(chisqr):.2e} (less is better)",
    xaxis2_title="m/z, Th",
    yaxis2_title="Counts, a.u."
)

# Show the figure
fig.show(config=plotly_config)

In [ ]:
fig = create_error_plot(fitted_pos, fitted_hei, positions, heights)
fig.show(config=plotly_config)

Most of the detected peaks are very close to the GT, both in m/z values and heights.
Now let's look at the undetected peaks and figure out what is special about them. 

In [ ]:
close_pos_inds = find_closest_indices(fitted_pos, positions)
undetected_mask = np.ones(heights.shape, dtype=bool)
undetected_mask[close_pos_inds] = False

fig, axes = plt.subplots(ncols=2, figsize=(10,3))

axes[0].hist(heights[undetected_mask], bins=50)
axes[0].set(
    yscale="log",
    title="Undetected peak height histogram",
    xlabel="Peak height",
    ylabel="Counts"
)

axes[1].plot(sorted(heights[undetected_mask]), marker=".")
axes[1].set(
    yscale="log",
    title="Undetected peak heights",
    xlabel="Peak height index",
    ylabel="Peak height"
);

Majority of undetected peaks have height of 10, close to the baseline. 

# Currently unused code

In [ ]:
# def gaussian_peak(x, shift, sigma, height=1):
#     """Generate a Gaussian peak.

#     :param x: The input values at which the Gaussian peak will be evaluated.
#     :type x: numpy array
#     :param shift: The center of the Gaussian peak
#     :type shift: float
#     :param sigma: The standard deviation of the Gaussian peak, controlling its width.
#     :type sigma: float
#     :return: The Gaussian peak evaluated at the input values x.
#     :type sigma: height, default 1
#     :return: Peak height.
#     :rtype: numpy array
#     """

#     return height * np.exp(-0.5 * ((x - shift) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

In [ ]:
# def generate_segment_lengths(num: int, params: dict) -> tuple:
#     """
#     Generate segment lengths and required number of sectors for each length.

#     :param num: The total number of segments to generate.
#     :type num: int
#     :param params: sector generation parameters
#     :type params: dict
#     :return: A tuple containing the lengths and corresponding amount of segments.
#     :rtype: tuple
#     """
#     if params["ms"] == "orbi":
#         # Sample uniformely distributed length values
#         uniform_samples = np.random.uniform(
#             low=min(params["lens"]), high=max(params["lens"]), size=num
#         )
#         # Convert lengths to int
#         uniform_samples = uniform_samples.astype("int")

#         # Get a histogram of uniformely distributed lengths
#         gen_bins, gen_edges = np.histogram(uniform_samples, bins=10)

#         # Apply inverse distribution
#         samples = gen_bins * inverse_x(gen_edges[1:], popt[0], popt[1])

#         # Return lengths and the required amount of sectors for each length
#         return gen_edges[1:], samples

In [ ]:
# # NOTE The difference in resolution within one sector is ~0.05 %
# # TODO peak should fully fit into the segment
# # TODO estimate max number of peaks that may be into the segment depending on its length
# # TODO maybe remove array computations for FWHM and sigma
# # TODO calculate one mean resolution for the whole segment


# def generate_orbi_segment(
#     start_mz: float,
#     length: int,
#     params: dict,
#     resolution,
# ) -> tuple:

#     mz_scale = [start_mz]
#     for _ in range(length-1):
#         # Resolution at this m/z
#         res = resolution(mz_scale[-1])
#         # FWHM
#         fwhm = mz_scale[-1] / res
#         # Step to the next mz value
#         dmz = fwhm / params["points_per_fwhm"]
#         # Add next mz value
#         mz_scale.append(mz_scale[-1] + dmz)

#     # Convert to numpy array
#     mz_scale = np.array(mz_scale)

#     # Random number of peaks, up to params["max_peaks"] 
#     num_of_peaks = np.random.randint(1, params["max_peaks"] + 1)

#     # Peak positions, up to num_of_peaks
#     peak_pos_indices = np.random.randint(0, len(mz_scale) - 1, num_of_peaks)
#     peak_pos = mz_scale[peak_pos_indices]

#     # Random peak heights
#     peak_heights = np.random.uniform(
#         min(params["heis"]), max(params["heis"]), num_of_peaks
#     )

#     # Calculate standart deviations for gaussian peaks
#     sigmas = peak_pos / resolution(peak_pos) / SIGMA_MULTIPLIER

#     print(peak_pos / resolution(peak_pos))
#     print(mz_scale.min(), mz_scale.max())

#     # Create intensity array
#     spec = np.zeros_like(mz_scale)
#     for i in range(num_of_peaks):
#         spec += gaussian_peak(mz_scale, peak_pos[i], sigmas[i])

#     # Save ground truth as dict
#     gt = {
#         "num_of_peaks": num_of_peaks,
#         "positions": mz_scale[peak_pos_indices],
#         "heights": peak_heights,
#     }

#     return (mz_scale, spec), gt


# def generate_orbi_segments(num: int, params: dict) -> list:
#     peakshape = gen_peak_shape(x, "orbi")
#     match params["ms"]:
#         case "orbi":
#             # Resolution function with a=1.725e6+-5%
#             resolution = partial(
#                 res_fun_orbi, a=1.725e6 * (1 + np.random.uniform(-0.05, 0.05))
#             )
#         case "tof":
#             resolution = res_fun_tof
#         case _:
#             raise ValueError("Unknown ms value in generation parameters")
#     segments = []
#     for _ in range(num):
#         gen_data = generate_orbi_segment()
#         segment = (gen_data[0], gen_data[1])
#         segments.append()